# YOLO26 real-time traffic monitoring on CPU

Running YOLO26 ONNX model on a CPU. Processing and displaying frames in real-time. Select a task and click run to get started.

- Detect: Frame-by-frame object detection.
- Track: Tracking objects across frames and assigning IDs.
- Speed Detection: Tracking objects and estimating their speed in km/h.

In [1]:
from ultralytics import YOLO
from pathlib import Path

onnx_model = "yolo26n.onnx"
onnx_path = Path.cwd() / onnx_model

if not onnx_path.exists():
    model = YOLO("yolo26n.pt")  # yolo26 nano model

    onnx_model = model.export(
        format="onnx",
        opset=12,  # OpenCV 5 need opset >= 12
        imgsz=640,  # fixed square shape
        dynamic=False,  # Avoid dynamic-shape parsing issue
        simplify=True,  # fold constants for cleaner graph
        nms=False,  # not needed anymore with YOLO26
    )

In [2]:
from typing import Any
import cv2
from PIL import Image
import numpy as np
from ultralytics import YOLO
import supervision as sv
from supervision.assets import download_assets, VideoAssets
from trackers import ByteTrackTracker
from types import GeneratorType
from ultralytics.engine.results import Results
from pathlib import Path
from collections import  defaultdict, deque


class YOLOInference:
    """
    A utility class for getting YOLO detection results.
    """

    def __init__(
        self, model: str = "yolo26n.onnx", device: str = "cpu", conf_thresh: float = 0.25
    ) -> None:

        self.device = device
        self.conf = conf_thresh

        # Load model
        self.model = YOLO(model, task="detect")
        self.model.overrides["device"] = self.device

        self.names = self.model.names

    @staticmethod
    def _to_sv_detections(preds: Results) -> sv.Detections:
        inputs = {
            "xyxy": preds.boxes.xyxy.cpu().numpy(),
            "class_id": preds.boxes.cls.cpu().numpy().astype(int),
            "confidence": preds.boxes.conf.cpu().numpy(),
        }
        
        return sv.Detections(**inputs)
    
    def detect_frame(self, frame: np.ndarray) -> Results:
        return self.model.predict(frame, conf=self.conf)[0]

    def track(
        self, source: Any, stream: bool = False, persist: bool = False, **kwargs
    ) -> Results | GeneratorType:
        preds = self.model.track(source, stream=stream, persist=persist, **kwargs)
        if isinstance(preds, GeneratorType):
            return preds
        return preds[0]

    def sv_detections(self, frame: np.ndarray) -> sv.Detections:
        preds = self.detect_frame(frame)
        return self._to_sv_detections(preds)


INPUT_VIDEO = download_assets(VideoAssets.VEHICLES)

# Each frame is resized at inference time
INPUT_SIZE = (640, 360)

# Source and target regions of interest for perspective transform needed for speed detection;
# transforming the source ROI to a region of size road width x road length
SOURCE = np.array([
    [208, 131],
    [383, 133],
    [856, 362],
    [-94, 360],
])
TARGET_WIDTH = 25       # assuming the road is 25m wide
TARGET_HEIGHT = 250     # assuming the source ROI captures 250 meters of the road
TARGET = np.array([
    [0, 0],
    [TARGET_WIDTH - 1, 0],
    [TARGET_WIDTH - 1, TARGET_HEIGHT - 1],
    [0, TARGET_HEIGHT - 1],
])

class PerspectiveTransform:
    """Turning xy coordinates from a frame into actual coordinates on the road"""

    def __init__(self, source: np.ndarray, target: np.ndarray) -> None:
        source = source.astype(np.float32)
        target = target.astype(np.float32)
        self.transform_mat = cv2.getPerspectiveTransform(source, target)

    def transform(self, points: np.ndarray) -> np.ndarray:
        if points.size == 0:
            return points
        transformed_points = cv2.perspectiveTransform(points.reshape(-1, 1, 2), self.transform_mat)
        return transformed_points.reshape(-1, 2)


class ProcessFrame:
    """
    Contains the methods for the computer vision tasks.
    """

    def __init__(self, input_video: str | Path, polygon_roi: np.ndarray | None = None, **yolo_kwargs):

        self.video_info = sv.VideoInfo.from_video_path(input_video)
        
        self.perspective_transform = PerspectiveTransform(SOURCE, TARGET)

        self.yolo_inference = YOLOInference(**yolo_kwargs)
        self.byte_track = ByteTrackTracker(frame_rate=self.video_info.fps)
        self.box_annot = sv.BoxAnnotator()
        self.label_annot = sv.LabelAnnotator()
        self.trace_annot = sv.TraceAnnotator(
            trace_length=self.video_info.fps,
            position=sv.Position.BOTTOM_CENTER,
        )
        self.polygon_roi = polygon_roi
        self.coordinates = defaultdict(lambda: deque(maxlen=int(self.video_info.fps)))
    
    def _track_detections(self, frame: np.ndarray) -> sv.Detections:
        detections = self.yolo_inference.sv_detections(frame)
        
        if self.polygon_roi is not None:
            polygon_zone = sv.PolygonZone(self.polygon_roi)
            detections = detections[polygon_zone.trigger(detections)]
        
        return self.byte_track.update(detections)
    
    def track_frame(self, frame: np.ndarray, *args) -> np.ndarray:
        detections = self._track_detections(frame)
        labels = [
            f"{self.yolo_inference.names[int(class_id)]} #{tracker_id}"
            for class_id, tracker_id in zip(detections.class_id, detections.tracker_id)
        ]
        annot_frame = frame.copy()
        if self.polygon_roi is not None:
            annot_frame = sv.draw_polygon(scene=annot_frame, polygon=self.polygon_roi)

        annot_frame = self.box_annot.annotate(scene=annot_frame, detections=detections)
        return self.label_annot.annotate(scene=annot_frame, detections=detections, labels=labels)

    def track_frame_for_speed_detection(self, frame: np.ndarray, *args) -> np.ndarray:
    
        detections = self._track_detections(frame)
        points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        transformed_points = self.perspective_transform.transform(points)
        
        for id, [x, y] in zip(detections.tracker_id, transformed_points):
            self.coordinates[id].append(y)

        labels = []
        for id in detections.tracker_id:
            if len(self.coordinates[id]) < self.video_info.fps / 2:
                labels.append(f"#{id}")
            else:
                # calculate speed
                coordinate_start = self.coordinates[id][-1]
                coordinate_end = self.coordinates[id][0]
                distance = abs(coordinate_start - coordinate_end)
                time = len(self.coordinates[id]) / self.video_info.fps
                speed = distance / time * 3.6
                labels.append(f"{int(speed)} km/h")
        
        annot_frame = frame.copy()
        if self.polygon_roi is not None:
            annot_frame = sv.draw_polygon(scene=annot_frame, polygon=self.polygon_roi)
        
        annot_frame = self.box_annot.annotate(scene=annot_frame, detections=detections)
        annot_frame = self.label_annot.annotate(scene=annot_frame, detections=detections, labels=labels)
        annot_frame = self.trace_annot.annotate(scene=annot_frame, detections=detections)

        return annot_frame
    
    def detect_frame(self, frame: np.ndarray, *args) -> np.ndarray:
        detections = self.yolo_inference.sv_detections(frame)
        labels = [
            f"{self.yolo_inference.names[int(class_id)]}" for class_id in detections.class_id
        ]
        annot_frame = frame.copy()
        annot_frame = self.box_annot.annotate(scene=annot_frame, detections=detections)
        return self.label_annot.annotate(scene=annot_frame, detections=detections, labels=labels)



[2026-08-23 16:19:06] [INFO] supervision.assets.downloader - Downloading vehicles.mp4 assets


  0%|          | 0/35345757 [00:00<?, ?it/s]

In [3]:
from ipywidgets import widgets, Image, IntProgress, Button, VBox, HBox
from IPython.display import display
import time
import plotly.graph_objects as go

video_info = sv.VideoInfo.from_video_path(INPUT_VIDEO)

method_dropdown = widgets.Dropdown(
    options=["detect", "track", "speed_detection"],
    value="detect",
    description="Task:",
)

run_btn = Button(description="Run", button_style="primary")
preview = Image(format="jpeg", width=640)
progress = IntProgress(min=0, max=video_info.total_frames)

fig = go.FigureWidget(
    go.Indicator(
        mode = "gauge+number",
        value = 0.0,
        title = {'text': "Average Inference Time (ms)"},
    )
)

display(VBox([HBox([method_dropdown, run_btn]), preview, progress, fig]))

method_map = {
    "speed_detection": ProcessFrame(INPUT_VIDEO, SOURCE, model=onnx_model, conf_thresh=0.3).track_frame_for_speed_detection,
    "track": ProcessFrame(INPUT_VIDEO, SOURCE, model=onnx_model, conf_thresh=0.3).track_frame,
    "detect": ProcessFrame(INPUT_VIDEO, model=onnx_model, conf_thresh=0.5).detect_frame,
}

frame_period = 1 / video_info.fps

def on_run(_):
    inference_times = []

    progress.value = 0
    cap = cv2.VideoCapture(INPUT_VIDEO)
    callback = method_map[method_dropdown.value]
    start_time = time.perf_counter()
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # resize to fixed 640x360
            resized_frame = cv2.resize(frame, INPUT_SIZE, interpolation=cv2.INTER_AREA)

            start_inference = time.perf_counter()
            annot = callback(resized_frame)
            inference_times.append(time.perf_counter() - start_inference)
            
            ok, buf = cv2.imencode(".jpg", annot)
            if ok:
                preview.value = buf.tobytes()
            progress.value += 1
            
            start_time += frame_period
            remainder = start_time - time.perf_counter()
            if remainder > 0:
                time.sleep(remainder)
            else:
                start_time = time.perf_counter()
    finally:
        cap.release()

    fig.data[0].value = (
        sum(inference_times) / max(len(inference_times), 1) * 1000
    )

run_btn.on_click(on_run)

Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.29.0 with CPUExecutionProvider
Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.29.0 with CPUExecutionProvider
Loading yolo26n.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.29.0 with CPUExecutionProvider
